# 05, Training diagnostics

Per `plan.md` §11 step 7. Read `results/logs/<name>/*.json` (per-fold
training history written by `scripts/train_logo.py`) and render
loss curves, per-fold val PR-AUC, and the metric table from
`results/ablations.csv`.

Run this after `python scripts/train_logo.py --name v1_baseline`
completes.

In [ ]:
%matplotlib inline
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
REPO = Path('..').resolve()
ABLATION = 'v1_baseline'  # change me

csv_path = REPO / 'results' / 'ablations.csv'
log_dir = REPO / 'results' / 'logs' / ABLATION
ablations = pd.read_csv(csv_path)
rows = ablations[ablations['name'] == ABLATION].sort_values('fold')
rows

## Per-fold loss curves

In [ ]:
histories = {}
for log_path in sorted(log_dir.glob('*.json')):
    fold = log_path.stem
    with log_path.open() as f:
        obj = json.load(f)
    histories[fold] = obj

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for fold, obj in histories.items():
    # The train_logo script doesn't currently log every-epoch history into the per-fold json,
    # only the final row + best PR-AUC. Loss curves come from the checkpoint's extras dict.
    ckpt_path = REPO / 'results' / 'checkpoints' / ABLATION / f'{fold}.pt'
    if not ckpt_path.exists():
        continue
    import torch
    ck = torch.load(ckpt_path, map_location='cpu')
    history = ck.get('extras', {}).get('history', [])
    if not history:
        continue
    epochs = [h['epoch'] for h in history]
    axes[0].plot(epochs, [h['train_loss'] for h in history], label=fold, alpha=0.7)
    axes[1].plot(epochs, [h['val_pr_auc'] for h in history], label=fold, alpha=0.7)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train loss'); axes[0].set_yscale('log')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val PR-AUC')
for ax in axes:
    ax.legend(fontsize=6, ncol=3)
fig.tight_layout()

## Per-fold metrics

In [ ]:
from hishells.eval import aggregate_folds
agg = aggregate_folds(rows.to_dict('records'))
for k, v in agg.items():
    print(f'{k:30s} {v}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(rows['fold'], rows['recall_at_op'])
axes[0].axvline(0.70, color='red', linestyle='--', label='target 0.70')
axes[0].set_xlabel('recall_at_op')
axes[0].legend()
axes[1].barh(rows['fold'], rows['fp_per_galaxy_at_op'])
axes[1].axvline(5, color='red', linestyle='--', label='target ≤ 5')
axes[1].set_xlabel('fp_per_galaxy_at_op')
axes[1].legend()
fig.tight_layout()

## Pass/fail vs proposal targets

Per §6.1: project succeeds if mean recall CI lower bound ≥ 0.70 *and*
mean FP/galaxy CI upper bound ≤ 5. The aggregate above prints both
as `passes_recall_target` / `passes_fp_target` for the chosen ablation.